In [22]:
from embedder import Embedder

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"
v1 = embed.encode(q1)

## Q1. Embedding a query

In [23]:
v1[0]

np.float64(-0.02058203437252893)

Answer : -0.02

In [24]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

## Q2. Cosine similarity

In [25]:
q2 = [doc["content"] for doc in documents if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"][0]
q2

'# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done so far is ex

In [26]:
v2 = embed.encode(q2)
v2

array([-6.69166283e-03, -1.96193582e-02, -1.35708113e-02,  2.79713869e-02,
        6.41942580e-02,  2.62990537e-02,  8.73512842e-03,  7.13342675e-03,
       -4.91385317e-02, -1.85060074e-02, -1.84053750e-02,  5.55403134e-02,
        1.10243809e-02, -2.03783775e-02, -1.14009158e-01,  1.41751117e-02,
       -1.42544043e-02,  4.95837365e-02,  8.09291555e-02,  4.17696865e-02,
       -2.95922966e-02,  4.67177638e-03,  4.05626478e-02, -3.90501575e-02,
        2.19090988e-03,  5.23046187e-02, -1.01323563e-02, -5.98989344e-02,
        4.16823513e-02,  5.88778963e-04,  3.38647085e-02,  2.94847756e-02,
        1.86943295e-02,  1.83957288e-01, -9.16205452e-02,  4.11525882e-02,
       -5.42344547e-02, -1.79471887e-02, -9.08646517e-02, -1.80611156e-02,
       -1.64216793e-02,  8.13043309e-03, -5.24594208e-02,  6.31449599e-02,
        4.37100755e-02,  3.84262345e-02, -2.89416559e-02, -6.02210131e-02,
        7.83269663e-02, -7.00202664e-02, -1.00836878e-01, -1.33440449e-02,
       -1.71584582e-02, -

In [27]:
v1.dot(v2)

np.float64(0.36107026789538205)

Answer : 0.36 -> 0.37

## Q3. Chunking and search by hand

In [7]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-28 16:34:49--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 888 [text/plain]
Saving to: ‘ingest.py.1’

ingest.py.1         100%[===================>]     888  --.-KB/s    in 0s      

2026-06-28 16:34:49 (42.1 MB/s) - ‘ingest.py.1’ saved [888/888]



In [28]:
import numpy as np
from tqdm.auto import tqdm
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [29]:
texts = [chunk["content"] for chunk in chunks]

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [30]:
scores = X.dot(v1)
scores

array([ 3.15187705e-01,  2.01479593e-01,  5.90559560e-02, -7.67661858e-02,
        1.18452480e-01, -1.41697042e-01, -2.81406552e-02, -4.65669225e-02,
       -2.06994704e-02, -6.07744087e-02,  2.13273853e-01,  8.87601799e-02,
       -1.97269351e-02,  3.11630016e-01,  5.51034674e-01,  2.04008048e-01,
        2.12515842e-01,  1.93649180e-01,  2.51961293e-01,  1.31078643e-01,
        2.59120579e-01,  7.63816008e-02,  9.59193707e-02,  9.81472975e-03,
       -3.59106882e-02,  1.01211577e-02,  1.10786937e-01, -9.90259208e-02,
       -3.71170151e-02,  7.59057570e-02, -3.35340540e-02,  8.86841309e-03,
        1.02636405e-01,  6.89614876e-02,  1.29408856e-01,  2.57709091e-01,
        3.23680614e-01,  1.06350075e-01,  5.61891367e-02,  2.34017457e-01,
        1.97954387e-01,  9.64296290e-02,  1.93709917e-01,  2.16719271e-01,
        3.48340456e-01,  5.10906092e-02,  2.05212837e-01,  1.05416170e-01,
       -3.25432514e-02,  4.94665548e-02,  2.38574865e-01, -3.44207108e-02,
        1.82165438e-01,  

In [32]:
idx = np.argmax(scores)
chunks[idx]


{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

Answer :  'filename': '02-vector-search/lessons/07-sqlitesearch-vector.md'}

## Q4. Vector search with minsearch

In [49]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

q4 = "What metric do we use to evaluate a search engine?"
results = vindex.search(embed.encode(q4), num_results=5)

In [ ]:
results[0]

{'start': 0,
 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

Answer : 04-evaluation/lessons/05-search-metrics.md

## Q5. Text search vs vector search

In [142]:
from minsearch import Index

index = Index(
    text_fields=["content"],
)

index.fit(documents)

q5 = "How do I store vectors in PostgreSQL?"

text_search_results = index.search(
    q5,
    num_results=5
)

text_search_results

[{'content': '# Embeddings\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=kJOlW1HeMp4&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nBefore we can do vector search, we need to turn our text into vectors.\nWe call this process embedding: we embed text into a vector space. The\nvectors we get back are also called "embeddings."\n\n## Word embeddings and sentence embeddings\n\nThis idea comes from\n[word2vec](https://en.wikipedia.org/wiki/Word2vec). The model learns to\nplace words as points in a multi-dimensional space. Words with similar\nmeanings land close to each other.\n\nImagine a 2D space where "enroll" and "join" are near each other and\n"Docker" is far away:\n\n```text\n        · enroll\n       · join\n                   · Docker\n```\n\nThe same idea works for entire sentences:\n\n```text\nQ1: "I just discovered the course. Can I still join it?"\nQ2: "I just found out about the program. Can I still enroll?"\n\nThese two are close - they mean the same thing.\n\nQ3: "H

In [143]:
vector_search_results = vindex.search(embed.encode(q5), num_results=5)

In [144]:
set([r["filename"] for r in vector_search_results])

{'02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md'}

In [145]:
set([r["filename"] for r in text_search_results])

{'02-vector-search/lessons/01-intro.md',
 '02-vector-search/lessons/02-embeddings.md',
 '02-vector-search/lessons/03-embeddings-dataset.md',
 '02-vector-search/lessons/10-next-steps.md',
 '03-orchestration/lessons/05-rag.md'}

Answser : 02-vector-search/lessons/08-pgvector.md

## Q6. Hybrid search


In [136]:
q6 = "How do I give the model access to tools?"

# keyword search
keyword_search = index = Index(
    text_fields=["content"],
).fit(chunks).search(
    q6,
)

# vector search
vector_search = VectorSearch().fit(X, chunks).search(embed.encode(q6))

In [137]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [138]:
results = rrf([vector_search, keyword_search])

In [139]:
results[0]

{'start': 4000,
 'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function ca

In [140]:
vector_search[0:5]

[{'start': 2000,
  'content': 'wrong.\n\n## The project\n\nRAG solves these problems by giving the LLM relevant documents at\nquestion time. We don\'t hope the model memorized the answer. We\nretrieve the right information and hand it to the LLM, and the model\ngenerates a grounded response. This lets us inject knowledge the model\nnever saw during training. That\'s why RAG is still the most common way\npeople use LLMs in the industry.\n\nTo make this concrete, we build a FAQ agent for our course. A student\nasks something like "when does the course start?" and the agent answers\nfrom the FAQ data we prepared.\n\nThis module has two parts.\n\nIn Part 1 (the next 9 lessons) we will:\n\n- Understand what RAG is and how it works\n- Build a search engine over a real FAQ dataset\n- Write a prompt that combines the user\'s question with search results\n- Wire it all together into a working RAG pipeline\n- Split ingestion and query into separate processes\n\nIn Part 2, we make the pipeline ag

In [141]:
keyword_search[0:5]

[{'start': 0,
  'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can

Answer : 01-agentic-rag/lessons/01-intro.md